# 🍳 Smart Recipe Recommender System
### Hybrid Content-Based, Nutritional & Knowledge-Graph Optimization Engine
**Author:** Aryaman Srimali Jain  
**Domain:** Information Retrieval, Recommender Systems, Natural Language Processing  

---

## 1. Executive Summary & Problem Definition
Modern household kitchens and meal planning systems face two major challenges:
1. **Pantry Underutilization & Food Waste:** Users frequently have a basket of perishable ingredients (e.g. vegetables, dairy, grains) but lack immediate ideas for suitable recipes.
2. **Dietary & Health Misalignment:** Traditional recipe search engines match keywords blindly, without optimizing for dietary constraints (Vegetarian, High Protein, Diabetic Friendly) or nutritional objectives (Weight Loss, Muscle Gain, Balanced Diet).

### Core Technical Deliverables:
- **TF-IDF Ingredient Vectorization:** Inverse Document Frequency penalizes common pantry staples (salt, water, oil) while prioritizing distinctive primary ingredients (paneer, prawns, mushrooms).
- **Overlap & Missing Ingredient Tracking:** Quantifies exact pantry coverage percentage and lists remaining ingredients to buy.
- **Multi-Objective Health Scoring:** Dynamic macronutrient conditioning tailored to user fitness goals.
- **Knowledge Graph Formulation (Neo4j):** Interlinks ingredients, recipes, courses, and cuisines into an interconnected graph ontology.

## 2. Environment Setup & Data Ingestion

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load cleaned master dataset
data_path = Path('../data/recipes_master.csv') if Path('../data/recipes_master.csv').exists() else Path('data/recipes_master.csv')
df = pd.read_csv(data_path)
print(f"Knowledge Base Loaded: {df.shape[0]} curated recipes, {df.shape[1]} features")
df[['name', 'cuisine', 'course', 'diet', 'total_time_mins', 'calories', 'protein_g', 'rating']].head()

## 3. Exploratory Data Analysis (EDA)
Visualizing the distribution of cuisines, dietary categories, and meal courses:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top Cuisines
top_cuisines = df['cuisine'].value_counts().head(8)
ax1.barh(top_cuisines.index[::-1], top_cuisines.values[::-1], color='#3b82f6')
ax1.set_title('Top 8 Cuisines in Knowledge Base', fontweight='bold')
ax1.set_xlabel('Recipe Count')

# Dietary Preferences
top_diets = df['diet'].value_counts().head(5)
ax2.pie(top_diets.values, labels=top_diets.index, autopct='%1.1f%%', colors=['#10b981', '#f59e0b', '#ec4899', '#8b5cf6', '#06b6d4'])
ax2.set_title('Dietary Preference Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. NLP Feature Extraction: TF-IDF Inverted Frequency Weighting
Common pantry staples (e.g. *salt*, *water*, *oil*) appear in almost every recipe and convey zero semantic specificity. We employ TF-IDF vectorization to automatically downweight frequent staples while amplifying distinctive ingredients:

In [ ]:
vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b', max_features=1000)
tfidf_matrix = vectorizer.fit_transform(df['ingredients_clean'].fillna(''))
idf_series = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

print("Lowest IDF Ingredients (Universal Staples - Penalized):")
print(idf_series.sort_values(ascending=True).head(6).round(2))

print("\nHighest IDF Ingredients (Distinctive Anchors - Amplified):")
print(idf_series.sort_values(ascending=False).head(6).round(2))

## 5. Recommender Engine Implementation
Instantiating the production `RecipeRecommender` engine:

In [ ]:
sys.path.append('..')
from src.recommender import RecipeRecommender

engine = RecipeRecommender(data_path)
print("Recommender Engine successfully compiled!")

## 6. Test Scenario A: Quick Weeknight Vegetarian Dinner
**Pantry Available:** `paneer, tomato, capsicum, onion, cumin, butter`  
**Constraints:** Vegetarian, Max Cook Time = 40 mins, Goal = Balanced

In [ ]:
results_a = engine.recommend(
    ingredients="paneer, tomato, capsicum, onion, cumin, butter",
    diet="Vegetarian",
    max_time_mins=40,
    diet_goal="balanced",
    top_k=3
)

for i, r in enumerate(results_a, 1):
    print(f"#{i} {r['name']} | Time: {r['total_time_mins']}m | Rating: {r['rating']}")
    print(f"   Score: {r['composite_score']} | Matched: {r['matched_ingredients']}")
    print(f"   Missing to Buy: {r['missing_ingredients']}")
    print(f"   Nutrition: {r['nutrition']['calories']} kcal, {r['nutrition']['protein_g']}g protein")
    print('-' * 70)

## 7. Test Scenario B: High-Protein Post-Workout Meal (Weight Loss Goal)
**Pantry Available:** `chicken, garlic, ginger, spinach, lemon, olive oil`  
**Constraints:** Diet Goal = `weight_loss` (maximizing protein-to-calorie ratio)

In [ ]:
results_b = engine.recommend(
    ingredients="chicken, garlic, ginger, spinach, lemon, olive oil",
    diet="All",
    diet_goal="weight_loss",
    top_k=3
)

for i, r in enumerate(results_b, 1):
    print(f"#{i} {r['name']} | Time: {r['total_time_mins']}m | Cuisine: {r['cuisine']}")
    print(f"   Calories: {r['nutrition']['calories']} kcal | Protein: {r['nutrition']['protein_g']}g | Fiber: {r['nutrition']['fiber_g']}g")
    print(f"   Matched: {r['matched_ingredients']}")
    print('-' * 70)

## 8. Knowledge Graph Architecture (Neo4j Cypher Schema)
In addition to vector similarity, recipes and ingredients can be modeled as a property graph:

In [ ]:
from src.neo4j_graph import RecipeKnowledgeGraph

cypher_example = RecipeKnowledgeGraph.get_recipe_query_cypher(['paneer', 'tomato', 'spinach'], min_overlap_ratio=0.5)
print("Sample Cypher Query for Graph-based Retrieval:")
print(cypher_example)

## 9. Conclusion & Key Takeaways
1. **Semantic Density vs Staple Noise:** Standard word counting fails on food recipes due to extreme staple prevalence (salt in 94% of recipes). TF-IDF successfully isolates recipe-defining ingredients.
2. **Pantry Coverage Metric:** By separating *matched* from *missing* ingredients, the system empowers real users to cook immediately or buy only 1-2 complementary items.
3. **Dynamic Health Alignment:** Conditioning ranking on macronutrient vectors ensures recommendations support the user's active lifestyle goals (Weight Loss vs Muscle Gain).